# Finantal-LM — Colab bootstrap (one-time setup)

Run this once per Colab session. It mounts Drive, clones the code repo, installs dependencies, sets the data paths, and verifies the data is present.

**Expected Drive layout** (`MyDrive/finantal_data/`):
```
finantal_data/
├── data/        pretrain_tokenized.jsonl, sft_tokenized_v2.jsonl
├── tokenizer/   finantal_tokenizer.model, .vocab
└── checkpoints/ pretrain/, sft/
```


In [ ]:
# Check the GPU (expect a Tesla T4)
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

In [ ]:
# Mount Google Drive (data + tokenizer + checkpoints live here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the code repo from GitHub
import os
REPO_URL = 'https://github.com/<YOUR_USERNAME>/finantal-lm.git'   # <-- EDIT
REPO_DIR = '/content/finantal-lm'
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull --ff-only
%cd $REPO_DIR

In [ ]:
# Install dependencies (torch already on Colab — not reinstalled)
!pip -q install -r requirements.txt

In [ ]:
# Point the code at the data on Drive. The default already matches this path,
# but we set it explicitly so it's obvious and overridable.
import os
os.environ['FINANTAL_DATA_ROOT'] = '/content/drive/MyDrive/finantal_data'
# Make the repo importable from notebook cells too
import sys; sys.path.insert(0, '/content/finantal-lm')
from config import paths as P
print(P.summary())

In [ ]:
# Verify the required assets exist on Drive before training
from config import paths as P
missing = P.verify_assets(require_tokenizer=True, require_pretrain=True, require_sft=True)
if missing:
    print('MISSING — upload these to Drive:')
    for m in missing: print('  -', m)
else:
    print('All assets present on Drive ✓')
    import json
    if os.path.exists(P.DATASET_STATS):
        print(json.dumps(json.load(open(P.DATASET_STATS)), ensure_ascii=False, indent=2))

Setup done. Now open **run_pretrain_colab.ipynb** or **run_sft_colab.ipynb**.